# <p style="text-align: center; font-size: 2.5rem">Notebook V</p>

<p style="text-align: center; font-size: 2rem">Plankton Density Imputation with Prophet</p>

This notebook uses Facebook Prophet to fit time series models on sparse COPEPOD plankton data per spatial region, impute continuous density values, and merge the results into the strandings dataset.

**Prerequisites**: Run `01_c_plankton_data_clean.ipynb` first to generate `copepod_dataset_se_coast.parquet`.


## Notebook Setup


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import logging

# Enable logging from the plankton module
logging.basicConfig(level=logging.INFO)

from se_coast_strandings.transformations import make_dt_col
from se_coast_strandings import make_degrees
from se_coast_strandings.contextual_data.plankton_abundance import (
    assign_region,
    prepare_prophet_series,
    fit_and_impute,
    build_plankton_lookup,
    merge_plankton_to_strandings,
)

In [ ]:
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
FIGURES_DIR = Path("../figures")
FIGURES_DIR.mkdir(exist_ok=True)

# Actual plankton count-per-volume column (renamed from VALUE-per-volu in 01_c)
VALUE_COL = "count"

# Time series frequency: 'W' = weekly, 'MS' = monthly start
FREQ = "W"

# ── Region tuning ────────────────────────────────────────────────────────────
# Must match DEGREES_PER_BAND used in notebooks 06/09/10 so regions align.
DEGREES_PER_BAND = 0.5
REGIONS = make_degrees(DEGREES_PER_BAND)

## Step 1: Load and Prepare the Plankton Data


In [ ]:
plankton_df = pd.read_parquet(PROCESSED_DIR / "copepod_dataset_se_coast.parquet")

# Recompute date from the correct lowercase columns (the stored 'date' column is corrupt)
plankton_df["date"] = make_dt_col(plankton_df["day"], plankton_df["month"], plankton_df["year"])

# Drop rows without valid dates or plankton values
plankton_df = plankton_df.dropna(subset=["date", VALUE_COL])

print(f"Plankton data: {plankton_df.shape[0]:,} rows")
print(f"Date range: {plankton_df['date'].min()} to {plankton_df['date'].max()}")
plankton_df.head()

## Step 2: Assign Spatial Regions

Each observation is assigned to a latitude-band region covering the SE coast.


In [ ]:
plankton_df["region"] = assign_region(plankton_df["lat"])

print("Observations per region:")
print(plankton_df["region"].value_counts().to_string())
print(f"\nUnassigned: {plankton_df['region'].isna().sum():,}")

## Step 3: Explore the Time Series per Region

Before fitting Prophet, let's see what the weekly aggregated series look like.


In [ ]:
fig, axes = plt.subplots(len(REGIONS), 1, figsize=(14, 3 * len(REGIONS)), sharex=True)

for ax, (label, _, _) in zip(axes, REGIONS):
    prophet_series = prepare_prophet_series(
        plankton_df, region=label, date_col="date", value_col=VALUE_COL, freq=FREQ
    )
    valid = prophet_series.dropna(subset=["y"])

    ax.plot(valid["ds"], valid["y"], "o", markersize=3, alpha=0.5, label="Observed")
    ax.set_title(f"Region: {label} ({len(valid)} data points)")
    ax.set_ylabel(VALUE_COL)
    ax.legend(loc="upper right")

axes[-1].set_xlabel("Date")
fig.suptitle("Raw Plankton Observations (weekly mean) by Region", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

## Step 4: Build the Full Plankton Lookup Table

This runs `build_plankton_lookup` which:
1. Assigns regions
2. Aggregates to weekly
3. Fits a Prophet model per region
4. Generates predictions over the full date range

This may take a few minutes.


In [ ]:
lookup = build_plankton_lookup(
    plankton_df,
    value_col=VALUE_COL,
    date_col="date",
    lat_col="lat",
    regions=REGIONS,
    freq=FREQ,
    forecast_end="2025-12-31",
)

print(f"Lookup table: {lookup.shape[0]:,} rows")
print(f"Regions: {lookup['region'].unique().tolist()}")
print(f"Date range: {lookup['ds'].min()} to {lookup['ds'].max()}")
lookup.head(10)

## Step 5: Visualize the Imputed Results

Overlay Prophet's imputed values on top of the original observations.


In [ ]:
fig, axes = plt.subplots(len(REGIONS), 1, figsize=(14, 4 * len(REGIONS)), sharex=True)

for ax, (label, _, _) in zip(axes, REGIONS):
    # Original observations
    raw = prepare_prophet_series(
        plankton_df, region=label, date_col="date", value_col=VALUE_COL, freq=FREQ
    ).dropna(subset=["y"])

    # Prophet forecast
    forecast = lookup[lookup["region"] == label]

    # Plot confidence interval
    ax.fill_between(
        forecast["ds"], forecast["yhat_lower"], forecast["yhat_upper"],
        alpha=0.15, color="steelblue", label="Confidence interval"
    )

    # Plot forecast line
    ax.plot(forecast["ds"], forecast["yhat"], "-", color="steelblue",
            linewidth=1, alpha=0.8, label="Prophet forecast")

    # Plot original observations
    ax.scatter(raw["ds"], raw["y"], s=10, color="red", alpha=0.6,
               zorder=5, label="Observed")

    ax.set_title(f"Region: {label}")
    ax.set_ylabel("Plankton density")
    ax.legend(loc="upper right", fontsize=8)

axes[-1].set_xlabel("Date")
fig.suptitle("Prophet Imputed Plankton Density vs Observed", y=1.02, fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "plankton_imputation_by_region.png", dpi=200, bbox_inches="tight")
plt.show()

## Step 6: Save the Lookup Table


In [ ]:
lookup.to_parquet(PROCESSED_DIR / "plankton_imputed_lookup.parquet", index=False)
print("Saved: data/processed/plankton_imputed_lookup.parquet")

## Step 7: Merge Plankton Density into Strandings

Load the strandings data and attach plankton density to each stranding event.


In [ ]:
# Load strandings
strandings_df = pd.read_excel(
    RAW_DIR / "UNC-DataRequest-01302026.xlsx", sheet_name="2015-2024"
)

# Create datetime column
strandings_df["mms_observation_dt"] = make_dt_col(
    strandings_df["Day of Observation"],
    strandings_df["Month of Observation"],
    strandings_df["Year of Observation"],
)

# Clean coordinates
strandings_df["Latitude"] = pd.to_numeric(strandings_df["Latitude"], errors="coerce")
strandings_df["Longitude"] = pd.to_numeric(strandings_df["Longitude"], errors="coerce")
strandings_df = strandings_df.dropna(subset=["Latitude", "Longitude", "mms_observation_dt"])

print(f"Strandings: {strandings_df.shape[0]:,} rows")
strandings_df.head()

In [ ]:
enriched = merge_plankton_to_strandings(
    strandings_df,
    lookup,
    stranding_date_col="mms_observation_dt",
    stranding_lat_col="Latitude",
)

print(f"Enriched strandings: {enriched.shape[0]:,} rows")
print(f"\nPlankton density coverage:")
print(f"  Has value: {enriched['plankton_density'].notna().sum():,}")
print(f"  Missing:   {enriched['plankton_density'].isna().sum():,}")
print(f"\nBy region:")
print(enriched["plankton_region"].value_counts().to_string())

In [ ]:
# Quick sanity check: plankton density distribution
fig, ax = plt.subplots(figsize=(10, 4))
enriched["plankton_density"].hist(bins=50, ax=ax, color="teal", edgecolor="black", linewidth=0.3)
ax.set_title("Distribution of Imputed Plankton Density across Strandings")
ax.set_xlabel("Plankton Density (imputed)")
ax.set_ylabel("Number of Strandings")
plt.tight_layout()
plt.show()

## Step 8: Save the Enriched Strandings Dataset


In [ ]:
enriched.to_parquet(PROCESSED_DIR / "strandings_with_plankton.parquet", index=False)
print("Saved: data/processed/strandings_with_plankton.parquet")